# Notebook 03 — Pipeline Dry Run

Runs the complete pipeline in **dry-run mode** (Layer 1 + Layer 2 only, no 3D generation) on 5 Tier 1 prompts. Validates that the graph retrieval → scene spec generation chain works end-to-end before committing GPU compute.

**No GPU required.** Set your ANTHROPIC_API_KEY below.

In [ ]:
import os, sys, json
REPO = 'ifc-graphrag-dt'
if os.path.exists(REPO): !cd {REPO} && git pull --quiet
else: !git clone https://github.com/aiwithprashant/ifc-graphrag-dt.git
sys.path.insert(0, REPO)

# ── Set your API key ──────────────────────────────────────────────────────────
# Either set here or in Colab Secrets (recommended)
import os
if 'ANTHROPIC_API_KEY' not in os.environ:
    os.environ['ANTHROPIC_API_KEY'] = 'your-key-here'  # replace before running

from pathlib import Path
from pipeline.layer1_retriever.ifc_graph_builder import IFCGraphBuilder
from pipeline.layer1_retriever.graph_embedder import GraphEmbedder
from pipeline.layer1_retriever.khop_traversal import KHopTraversal
from pipeline.layer2_spec_gen.scene_spec_generator import SceneSpecGenerator
from evaluation.stage_a.spec_evaluator import StageAEvaluator
from benchmark.dtah_bench import DTAHBench

IFC_PATH    = f'{REPO}/benchmark/ifc_reference_models/duplex.ifc'
GRAPH_CACHE = f'{REPO}/outputs/graphs/ifc_graph.json'
EMB_CACHE   = f'{REPO}/outputs/embedders/graph_embedder'
SPEC_OUT    = f'{REPO}/outputs/specs'
os.makedirs(SPEC_OUT, exist_ok=True)
print('Imports OK')

In [ ]:
# ── Cell 2: Load graph and embedder ──────────────────────────────────────────
if Path(GRAPH_CACHE).exists():
    G = IFCGraphBuilder.load_graph(GRAPH_CACHE)
else:
    builder = IFCGraphBuilder(IFC_PATH)
    G = builder.build()
    builder.save_graph(GRAPH_CACHE)

if Path(EMB_CACHE).exists():
    embedder = GraphEmbedder.load(EMB_CACHE)
else:
    embedder = GraphEmbedder()
    embedder.fit(G)
    embedder.save(EMB_CACHE)

print(f'Graph: {G.number_of_nodes()} nodes | Embedder: {len(embedder._node_ids)} nodes')

In [ ]:
# ── Cell 3: Initialise spec generator ────────────────────────────────────────
spec_config = {
    'llm_provider': 'anthropic',
    'model': 'claude-sonnet-4-20250514',
    'max_tokens': 2048,
    'temperature': 0.0,
    'output_dir': SPEC_OUT,
}
spec_gen  = SceneSpecGenerator(spec_config)
evaluator = StageAEvaluator()
print('Spec generator initialised.')

In [ ]:
# ── Cell 4: Dry run on 5 Tier 1 prompts ──────────────────────────────────────
bench   = DTAHBench(pilot_mode=True)
tier1   = bench.load_tier(1)[:5]  # 5 prompts for dry run
results = []

for p in tier1:
    pid, prompt, tier = p['id'], p['prompt'], p['tier']
    print(f'\n[{pid}] {prompt}')

    # Layer 1: retrieve
    seeds = embedder.retrieve_seeds(prompt, top_k=5)
    seed_ids = [s['node_id'] for s in seeds]
    trav = KHopTraversal(G, max_depth=1, bidirectional=True)  # Tier 1 → depth 1
    trav_result = trav.traverse(seed_ids)
    subgraph = trav_result.subgraph
    print(f'  Layer 1: {subgraph.number_of_nodes()} nodes, {subgraph.number_of_edges()} edges')

    # Layer 2: generate spec
    spec = spec_gen.generate(prompt=prompt, subgraph=subgraph,
                             prompt_id=pid, tier=tier, domain=p.get('domain','MEP'))
    print(f'  Layer 2: {len(spec.get("entities",[]))} entities, '
          f'{len(spec.get("relations",[]))} relations in spec')
    print(f'  Generation prompt: {spec.get("generation_prompt","")[:80]}...')

    results.append({'id': pid, 'prompt': prompt, 'spec': spec,
                    'subgraph_nodes': subgraph.number_of_nodes(),
                    'subgraph_edges': subgraph.number_of_edges()})

print(f'\n✓ Dry run complete: {len(results)} prompts processed')

In [ ]:
# ── Cell 5: Inspect a generated spec ─────────────────────────────────────────
print('Sample spec output for first prompt:')
sample = results[0]['spec'].copy()
# Remove bulky generation_prompt for display
print(json.dumps({k: v for k, v in sample.items() if k != 'generation_prompt'}, indent=2))

In [ ]:
# ── Cell 6: Stage A evaluation (self-consistency check) ──────────────────────
# Without annotated ground truth, we test self-consistency:
# A perfect pipeline should score ~1.0 when evaluated against itself

print('Stage A self-consistency scores (pred vs pred — should be ~1.0):')
print(f'{"ID":<15} {"E":>6} {"R":>6} {"A":>6} {"Total":>8}')
print('-' * 45)

for r in results:
    spec = r['spec']
    score = evaluator.evaluate(spec, spec, prompt_id=r['id'])
    print(f'{r["id"]:<15} {score.entity_score:>6.3f} {score.relation_score:>6.3f} '
          f'{score.attribute_score:>6.3f} {score.total:>8.3f}')

print('\nNote: Real Stage A scores require annotated ground-truth specs.')
print('Dry run validation complete. Ready for GPU experiments in Notebook 04.')

In [ ]:
# ── Cell 7: Save dry run results ─────────────────────────────────────────────
dry_run_summary = [{
    'id': r['id'],
    'prompt': r['prompt'],
    'subgraph_nodes': r['subgraph_nodes'],
    'subgraph_edges': r['subgraph_edges'],
    'spec_entities': len(r['spec'].get('entities',[])),
    'spec_relations': len(r['spec'].get('relations',[])),
    'spec_constraints': len(r['spec'].get('constraints',[])),
    'generation_prompt_len': len(r['spec'].get('generation_prompt','')),
} for r in results]

out_path = f'{REPO}/outputs/scores/dry_run_summary.json'
os.makedirs(os.path.dirname(out_path), exist_ok=True)
with open(out_path, 'w') as f:
    json.dump(dry_run_summary, f, indent=2)

print(f'Dry run summary saved: {out_path}')
print(json.dumps(dry_run_summary, indent=2))